In [ ]:
!git clone https://github.com/VasiliyMakshanchikov/stochastic-flow-stability-visual-noise.git
%cd stochastic-flow-stability-visual-noise/
!git checkout feature/inference-attempts

Cloning into 'stochastic-flow-stability-visual-noise'...
remote: Enumerating objects: 2234, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 2234 (delta 53), reused 74 (delta 48), pack-reused 2152 (from 2)
Receiving objects: 100% (2234/2234), 115.96 MiB | 37.70 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/stochastic-flow-stability-visual-noise
Branch 'feature/inference-attempts' set up to track remote branch 'feature/inference-attempts' from 'origin'.
Switched to a new branch 'feature/inference-attempts'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd GroundingDINO/
!pip uninstall -y timm
!pip install timm>=0.9.16
!pip install -r requirements.txt
!pip uninstall -y transformers tokenizers
!pip install transformers==4.35.2 tokenizers==0.15.2
!pip install ninja
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

/content/stochastic-flow-stability-visual-noise/GroundingDINO
Found existing installation: timm 1.0.26
Uninstalling timm-1.0.26:
  Successfully uninstalled timm-1.0.26
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.1 MB/s eta 0:00:00
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
%cd ..
!pip uninstall -y groundingdino
!rm -rf GroundingDINO

!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd GroundingDINO
!pip install -e .

/content/stochastic-flow-stability-visual-noise
Cloning into 'GroundingDINO'...
remote: Enumerating objects: 463, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 463 (delta 108), reused 90 (delta 90), pack-reused 316 (from 1)
Receiving objects: 100% (463/463), 12.88 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (239/239), done.
/content/stochastic-flow-stability-visual-noise/GroundingDINO
Obtaining file:///content/stochastic-flow-stability-visual-noise/GroundingDINO
  Preparing metadata (setup.py) ... done
  Running setup.py develop for groundingdino


In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
import tempfile

# Импорты для GroundingDINO
from groundingdino.util.inference import load_model
from groundingdino.datasets import transforms as T
from torchvision.transforms import functional as F
from torchvision import transforms as TF

import sys
sys.path.append(os.path.abspath('/content/stochastic-flow-stability-visual-noise/GroundingDINO'))
sys.path.append(os.path.abspath('/content/stochastic-flow-stability-visual-noise/GroundingDINO/groundingdino'))
from groundingdino.util.inference import load_model, load_image, predict

# Отключаем предупреждения токенизатора
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ========================
#  Конфигурация
# ========================



FRAME_DIR = "/content/stochastic-flow-stability-visual-noise/mall_dataset/frames"
OUTPUT_BASE = Path("/content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr")


# FRAME_DIR = "/content/stochastic-flow-stability-visual-noise/mall_dataset/pre_frames"
# OUTPUT_BASE = Path("/content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr_test")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = "/content/stochastic-flow-stability-visual-noise/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
WEIGHTS_PATH = "/content/drive/MyDrive/Диплом магистратура/groundingdino_swint_ogc.pth"

TEXT_PROMPT = "people"
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

In [ ]:
import re

results_dir = Path("/content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr_test")

# 1. Загружаем оригинал
original = np.load(results_dir / "original.npy")

# 2. Собираем все файлы гауссовского шума
gaussian_files = list(results_dir.glob("gaussian_var_*.npy"))

# 3. Извлекаем значения параметра var из имени
data = []
for f in gaussian_files:
    # Извлекаем число после "var_"
    match = re.search(r"var_([0-9.]+)\.npy", f.name)
    if match:
        var = float(match.group(1))
        counts = np.load(f)
        data.append((var, counts))

# 4. Сортируем по var
data.sort(key=lambda x: x[0])

# 5. Для каждого var вычисляем метрику (например, среднюю абсолютную ошибку counts)
# for var, counts in data:
#     mae = np.mean(np.abs(counts - original))
#     # print(f"var={var:.6f}, MAE={mae:.3f}")
#     print(counts)

data

[(0.001, array([22, 25, 23, 21, 20, 25, 24, 24, 21, 23])),
 (0.006444, array([22, 23, 23, 21, 22, 25, 24, 24, 21, 23])),
 (0.011889, array([22, 22, 23, 21, 21, 25, 24, 23, 21, 23])),
 (0.017333, array([21, 22, 23, 21, 20, 25, 24, 23, 20, 23])),
 (0.022778, array([21, 22, 24, 21, 19, 25, 24, 23, 21, 23])),
 (0.028222, array([22, 23, 23, 21, 20, 25, 24, 23, 21, 23])),
 (0.033667, array([22, 22, 23, 21, 20, 25, 24, 23, 21, 23])),
 (0.039111, array([22, 23, 23, 21, 20, 24, 24, 23, 20, 23])),
 (0.044556, array([22, 23, 23, 21, 19, 25, 24, 23, 20, 23])),
 (0.05, array([21, 22, 23, 21, 19, 25, 24, 23, 20, 23]))]

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используется устройство: {DEVICE}")

# ---------------------------
#  Загрузка модели на GPU
# ---------------------------
model = load_model(CONFIG_PATH, WEIGHTS_PATH)
model = model.to(DEVICE)
model.eval()

# ---------------------------
#  Функции добавления шума (в памяти)
# ---------------------------
def add_gaussian_noise(images, mean=0.0, var=0.01):
    noisy = []
    sigma = np.sqrt(var)
    for img in images:
        noise = np.random.normal(mean, sigma, img.shape).astype(np.float32)
        noisy_img = img.astype(np.float32) + noise
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        noisy.append(noisy_img)
    return noisy

def add_salt_pepper_noise(images, salt_prob=None, pepper_prob=None, prob=None):
    """
    Добавляет шум "соль и перец" к списку изображений.
    Либо указывайте salt_prob и pepper_prob по отдельности,
    либо используйте prob (тогда salt_prob = pepper_prob = prob).
    """
    if prob is not None:
        salt_prob = prob
        pepper_prob = prob
    if salt_prob is None:
        salt_prob = 0.01
    if pepper_prob is None:
        pepper_prob = 0.01

    noisy = []
    for img in images:
        out = img.copy()
        total_pixels = img.shape[0] * img.shape[1]   # количество пикселей

        # Соль (белые точки)
        num_salt = int(salt_prob * total_pixels)
        if num_salt > 0:
            rows = np.random.randint(0, img.shape[0], num_salt)
            cols = np.random.randint(0, img.shape[1], num_salt)
            out[rows, cols] = 255

        # Перец (чёрные точки)
        num_pepper = int(pepper_prob * total_pixels)
        if num_pepper > 0:
            rows = np.random.randint(0, img.shape[0], num_pepper)
            cols = np.random.randint(0, img.shape[1], num_pepper)
            out[rows, cols] = 0

        noisy.append(out)
    return noisy

def add_speckle_noise(images, scale=0.1):
    noisy = []
    for img in images:
        noise = np.random.randn(*img.shape) * scale
        noisy_img = img.astype(np.float32) + img.astype(np.float32) * noise
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        noisy.append(noisy_img)
    return noisy

def add_poisson_noise(images, scale=1.0):
    noisy = []
    for img in images:
        img_float = img.astype(np.float32) / 255.0
        noisy_float = np.random.poisson(img_float * scale) / scale
        noisy_img = np.clip(noisy_float * 255, 0, 255).astype(np.uint8)
        noisy.append(noisy_img)
    return noisy

def add_noise_batch(images, noise_type, **kwargs):
    if noise_type == "gaussian":
        return add_gaussian_noise(images, **kwargs)
    elif noise_type == "salt_pepper":
        return add_salt_pepper_noise(images, **kwargs)
    elif noise_type == "speckle":
        return add_speckle_noise(images, **kwargs)
    elif noise_type == "poisson":
        return add_poisson_noise(images, **kwargs)
    else:
        return images

# ---------------------------
#  Инференс через predict с временным файлом
# ---------------------------
def count_people_in_image(model, image_bgr, caption, box_threshold, text_threshold, device):
    """
    image_bgr: numpy array (H,W,3) в формате BGR (как читает OpenCV)
    """
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        tmp_path = tmp.name
        cv2.imwrite(tmp_path, image_bgr)
    try:
        _, image_tensor = load_image(tmp_path)   # load_image возвращает (image_source, image_tensor)
        boxes, logits, phrases = predict(
            model=model,
            image=image_tensor,
            caption=caption,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device=device
        )
        return len(boxes)
    finally:
        os.unlink(tmp_path)

# ---------------------------
#  Предзагрузка кадров
# ---------------------------
def load_all_frames(frame_dir):
    frame_files = sorted([f for f in os.listdir(frame_dir) if f.endswith('.jpg')])
    frames = []
    for fname in frame_files:
        img = cv2.imread(os.path.join(frame_dir, fname))
        if img is None:
            print(f"Предупреждение: не удалось прочитать {fname}")
            continue
        frames.append((fname, img))
    return frames

# ---------------------------
#  Обработка одного эксперимента
# ---------------------------
def process_experiment(frames, noise_type, param_name, param_value, fixed_params, output_path):
    # 1. Генерация зашумлённых изображений
    original_images = [img for _, img in frames]
    if noise_type is None:   # baseline
        noisy_images = original_images
    else:
        kwargs = fixed_params.copy()
        kwargs[param_name] = param_value
        noisy_images = add_noise_batch(original_images, noise_type, **kwargs)

    # 2. Инференс (через временные файлы)
    counts = []
    for img in tqdm(noisy_images, desc=f"→ {output_path.name}", leave=False):
        cnt = count_people_in_image(model, img, TEXT_PROMPT, BOX_THRESHOLD, TEXT_THRESHOLD, DEVICE)
        counts.append(cnt)

    counts = np.array(counts, dtype=int)
    np.save(output_path, counts)
    print(f"Сохранено: {output_path}")
    return counts

# ---------------------------
#  Запуск всех экспериментов
# ---------------------------
def run_experiments():
    print("Загрузка оригинальных кадров...")
    frames = load_all_frames(FRAME_DIR)
    if not frames:
        print("Нет кадров для обработки!")
        return

    # Baseline (без шума)
    baseline_path = OUTPUT_BASE / "original.npy"
    if not baseline_path.exists():
        print("\nОбработка baseline (без шума)...")
        process_experiment(frames, None, None, None, {}, baseline_path)
    else:
        print(f"Baseline уже существует: {baseline_path}")

    # Конфигурации экспериментов
    experiments_config = [
        {
            "noise_type": "gaussian",
            "param_name": "var",
            "values": np.linspace(0.001, 0.05, 10),
            "fixed_params": {"mean": 0.0}
        },
        {
            "noise_type": "salt_pepper",
            "param_name": "prob",
            "values": np.linspace(0.001, 0.05, 10),
            "fixed_params": {}
        },
        {
            "noise_type": "speckle",
            "param_name": "scale",
            "values": np.linspace(0.05, 0.8, 12),
            "fixed_params": {}
        },
        {
            "noise_type": "poisson",
            "param_name": "scale",
            "values": np.linspace(0.2, 2.0, 10),
            "fixed_params": {}
        }
    ]

    total_experiments = sum(len(cfg["values"]) for cfg in experiments_config)
    print(f"\nВсего экспериментов (без baseline): {total_experiments}")

    for cfg in experiments_config:
        noise_type = cfg["noise_type"]
        param_name = cfg["param_name"]
        values = cfg["values"]
        fixed = cfg["fixed_params"]

        for val in tqdm(values, desc=f"Тип шума: {noise_type}"):
            filename = f"{noise_type}_{param_name}_{val:.6f}.npy"
            out_path = OUTPUT_BASE / filename
            if out_path.exists():
                continue
            process_experiment(frames, noise_type, param_name, val, fixed, out_path)

    print("\nВсе эксперименты завершены!")

# ---------------------------
#  Выполнение
# ---------------------------
if __name__ == "__main__":
    import time
    start = time.perf_counter()
    run_experiments()
    end = time.perf_counter()
    print(f"Общее время выполнения: {end - start:.2f} сек")


Используется устройство: cuda


final text_encoder_type: bert-base-uncased


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Сохранено: /content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr/poisson_scale_1.600000.npy



Тип шума: poisson:  90%|█████████ | 9/10 [44:46<06:21, 381.37s/it]

Сохранено: /content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr/poisson_scale_1.800000.npy



Тип шума: poisson: 100%|██████████| 10/10 [59:41<00:00, 358.13s/it]

Сохранено: /content/drive/MyDrive/Диплом магистратура/groundingdino_results_parr/poisson_scale_2.000000.npy

Все эксперименты завершены!
Общее время выполнения: 3586.24 сек
